# 第 7 章 分類モデルの評価指標

正解率が役に立たない場面を出発点に、混同行列・適合率・再現率・F1・AUC を確かめます。

対応する記事: [第 7 章 分類モデルの評価指標（Python 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/python/ch07.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch07_metrics import *

## 正解率 99% の役立たずモデル

1000 人に 10 人が罹る病気を、**全員「陰性」と判定する** モデルです。正解率は 99% ですが、病人を 1 人も見つけられません。

**正解率は、陽性と陰性の数が偏っているときに壊れます。**

In [2]:
sick_labels = [1] * 10 + [0] * 990
always_healthy = [0] * 1000

matrix = confusion_matrix(sick_labels, always_healthy)
print(matrix)
print(f"正解率 {accuracy(matrix):.3f}")
print(f"再現率 {recall(matrix):.3f}  ← 病人を 1 人も見つけられていない")

ConfusionMatrix(true_positives=0, false_positives=0, false_negatives=10, true_negatives=990)
正解率 0.990
再現率 0.000  ← 病人を 1 人も見つけられていない


## 適合率と再現率はトレードオフ

逆に **全員を「陽性」と判定** すると、見逃しはゼロ（再現率 1.0）ですが、陽性と言った 1000 件のうち当たりは 10 件だけです。**片方の指標だけを追うと、必ずもう一方が壊れます。**

In [3]:
aggressive = confusion_matrix(sick_labels, [1] * 1000)

print(f"{'モデル':<12} {'正解率':>8} {'適合率':>8} {'再現率':>8} {'F1':>8}")
for name, m in [("全員陰性", matrix), ("全員陽性", aggressive)]:
    print(f"{name:<12} {accuracy(m):>8.3f} {precision(m):>8.3f} {recall(m):>8.3f} {f1_score(m):>8.3f}")

モデル               正解率      適合率      再現率       F1
全員陰性            0.990    0.000    0.000    0.000
全員陽性            0.010    0.010    1.000    0.020


## F1 は調和平均

適合率 1.0・再現率 0.1 のモデルは、算術平均なら 0.55 と「まあまあ」に見えます。**F1 は 0.18 です。片方が壊れているモデルを、平均で誤魔化させません。**

In [4]:
unbalanced = ConfusionMatrix(true_positives=1, false_positives=0, false_negatives=9, true_negatives=90)

print(f"適合率 {precision(unbalanced):.3f}  再現率 {recall(unbalanced):.3f}")
print(f"算術平均 {(precision(unbalanced) + recall(unbalanced)) / 2:.3f}")
print(f"F1       {f1_score(unbalanced):.3f}  ← 偏りを強く罰する")

適合率 1.000  再現率 0.100
算術平均 0.550
F1       0.182  ← 偏りを強く罰する


## F ベータで重視する側を選ぶ

病気の見逃しを避けたいなら `beta > 1`（再現率重視）、迷惑メール判定で誤検知を避けたいなら `beta < 1`（適合率重視）です。**指標そのものを目的に合わせて調整できます。**

In [5]:
sample = ConfusionMatrix(true_positives=3, false_positives=1, false_negatives=2, true_negatives=4)
print(f"適合率 {precision(sample):.3f}  再現率 {recall(sample):.3f}")

for beta in [0.5, 1.0, 2.0]:
    print(f"F{beta} = {f_beta_score(sample, beta=beta):.4f}")

適合率 0.750  再現率 0.600
F0.5 = 0.7143
F1.0 = 0.6667
F2.0 = 0.6250


## AUC は閾値に依存しない

AUC は **「陽性を陰性より高くランク付けできた組の割合」** です。確率が両極に分かれていても中央に固まっていても、**順位が同じなら AUC は同じ** になります。

In [6]:
cases = [
    ("完全な順位", [1, 1, 0, 0], [0.9, 0.8, 0.2, 0.1]),
    ("3/4 が正しい順", [1, 0, 1, 0], [0.8, 0.6, 0.4, 0.2]),
    ("情報なし", [1, 0, 0, 1], [0.8, 0.6, 0.4, 0.2]),
    ("完全に逆", [0, 0, 1, 1], [0.9, 0.8, 0.2, 0.1]),
]
for name, ls, ps in cases:
    print(f"{name:<16} AUC = {auc(ls, ps):.3f}")

print()
print("確率を圧縮しても AUC は変わらない:")
print(" 両極 ", auc([1, 1, 0, 0], [0.99, 0.98, 0.02, 0.01]))
print(" 中央 ", auc([1, 1, 0, 0], [0.55, 0.54, 0.46, 0.45]))

完全な順位            AUC = 1.000
3/4 が正しい順        AUC = 0.750
情報なし             AUC = 0.500
完全に逆             AUC = 0.000

確率を圧縮しても AUC は変わらない:
 両極  1.0
 中央  1.0


## 試してみる: 閾値を動かす

**閾値はモデルの性能ではなく、運用上の選択です。** 下げれば再現率が上がり、適合率が下がります。

In [7]:
labels = [1, 1, 0, 0]
probabilities = [0.9, 0.4, 0.6, 0.1]

print(f"{'閾値':>6} {'予測':<16} {'適合率':>8} {'再現率':>8}")
for threshold in [0.2, 0.5, 0.8]:
    predictions = predictions_at_threshold(probabilities, threshold)
    m = confusion_matrix(labels, predictions)
    print(f"{threshold:>6} {predictions!s:<16} {precision(m):>8.3f} {recall(m):>8.3f}")

    閾値 予測                    適合率      再現率
   0.2 [1, 1, 1, 0]        0.667    1.000
   0.5 [1, 0, 1, 0]        0.500    0.500
   0.8 [1, 0, 0, 0]        1.000    0.500
